## Gráficos

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_spark = spark.read.parquet("data/interim/01_processed.parquet")
pdf = df_spark.toPandas()

# Display basic info
print(f"Dataset loaded with {len(pdf)} rows and {len(pdf.columns)} columns.")
display(pdf.head())

In [ ]:
# Define mappings
map_sexo = {'1': 'Masculino', '2': 'Feminino'}
map_tabagismo = {'1': 'Nunca', '2': 'Ex-consumidor', '3': 'Sim', '4': 'Não avaliado', '8': 'Não se aplica', '9': 'Sem informação'}
map_alcool = map_tabagismo.copy()
map_racacor = {'1': 'Branca', '2': 'Preta', '3': 'Amarela', '4': 'Parda', '5': 'Indígena', '9': 'Sem informação'}
map_base_diagnostico = {'0': 'Óbito', '1': 'Clínico', '2': 'Clínico, investigação', '3': 'Cirurgia', '4': 'Marcadores bioquímicos', 
                        '5': 'Citologia', '6': 'Histologia de metástase', '7': 'Histologia de tumor primário', '9': 'Sem informação'}
map_exdiag = {'1': 'Nenhum', '2': 'Raio-X', '3': 'Mamografia', '4': 'Tomografia', '5': 'Ressonância', '6': 'Outros'}
map_pritrath = {'1': 'Nenhum', '2': 'Cirurgia', '3': 'Radioterapia', '4': 'Quimioterapia', '5': 'Hormonioterapia', '6': 'TMO', '7': 'Outro'}
map_estdfimt = {'1': 'Sem evidência de doença', '2': 'Remissão parcial', '3': 'Doença estável', '4': 'Progressão da doença', 
                '5': 'Recidiva', '6': 'Óbito', '9': 'Sem informação'}
map_tp_caso = {'1': 'Analítico', '2': 'Não analítico'}

# Apply mappings
pdf['sexo_label'] = pdf['sexo'].map(map_sexo)
pdf['tabagismo_label'] = pdf['historico_tabagismo'].map(map_tabagismo)
pdf['alcool_label'] = pdf['historico_alcoolismo'].map(map_alcool)
pdf['raca_cor_label'] = pdf['raca_cor'].map(map_racacor)
pdf['base_diag_label'] = pdf['base_diagnostico_mais_importante'].map(map_base_diagnostico)
pdf['exdiag_label'] = pdf['exames_relevantes_diagnostico'].map(map_exdiag)
pdf['pritrath_label'] = pdf['primeiro_tratamento_hospital'].map(map_pritrath)
pdf['tipo_caso_label'] = pdf['tipo_caso'].astype(str).str.strip().map(map_tp_caso).fillna('Sem informação')
pdf['status_vital_label'] = pdf['status_vital'].map({0: 'Óbito', 1: 'Vivo'})

# Date parts
pdf['ano_diagnostico'] = pd.to_datetime(pdf['data_primeiro_diagnostico']).dt.year
pdf['idade'] = pd.to_numeric(pdf['idade'], errors='coerce')

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x=pdf['status_vital_label'].value_counts(normalize=True).index, 
            y=pdf['status_vital_label'].value_counts(normalize=True).values, 
            palette='viridis')
plt.title('Gráfico status_vital')
plt.ylabel('Proporção')
plt.xlabel('Status Vital')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=pdf[pdf['data_obito'].isna()], x='ano_diagnostico', palette='viridis')
plt.title("Count of nan in data_obito by Year of data_primeiro_diagnostico")
plt.xlabel("Year of data_primeiro_diagnostico")
plt.ylabel("Count of nan in data_obito")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=pdf, x='sexo_label', ax=axes[0], palette='viridis', order=pdf['sexo_label'].value_counts().index)
axes[0].set_title('Sexo')

sns.countplot(data=pdf, x='raca_cor_label', ax=axes[1], palette='viridis', order=pdf['raca_cor_label'].value_counts().index)
axes[1].set_title('Raça/Cor')

sns.histplot(data=pdf, x='idade', bins=30, ax=axes[2], color='teal')
axes[2].set_title('Idade')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=pdf, x='tabagismo_label', ax=axes[0], palette='viridis', order=pdf['tabagismo_label'].value_counts().index)
axes[0].set_title('Histórico de Tabagismo')

sns.countplot(data=pdf, x='alcool_label', ax=axes[1], palette='viridis', order=pdf['alcool_label'].value_counts().index)
axes[1].set_title('Histórico de Alcoolismo')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.countplot(data=pdf, x='base_diag_label', ax=axes[0], palette='viridis', order=pdf['base_diag_label'].value_counts().index)
axes[0].set_title('Base Mais Importante do Diagnóstico')
axes[0].tick_params(axis='x', rotation=45)

sns.countplot(data=pdf, x='exdiag_label', ax=axes[1], palette='viridis', order=pdf['exdiag_label'].value_counts().index)
axes[1].set_title('Exames Relevantes do Diagnóstico')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(10, 5))
sns.countplot(data=pdf, x='pritrath_label', ax=axes, palette='viridis', order=pdf['pritrath_label'].value_counts().index)
axes.set_title('Primeiro Tratamento Recebido')
axes.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=pdf, x='idade', hue='status_vital_label', multiple='layer', bins=30, alpha=0.6, palette='viridis')
plt.title('Distribuição de Idade por Status Vital')
plt.xlabel('Idade')
plt.tight_layout()
plt.show()

In [ ]:
cross = pd.crosstab(pdf['tipo_caso_label'], pdf['status_vital_label'])
cross_prop = cross.div(cross.sum(axis=1), axis=0)
cross_prop.plot(kind='bar', stacked=True, figsize=(8, 5), colormap='viridis')
plt.title('Proporção de Óbito por Tipo de Caso')
plt.ylabel('Proporção')
plt.xlabel('Tipo de Caso')
plt.legend(title='Status Vital')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()